In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [2]:
import pandas as pd

df_orders = pd.read_csv("orders.csv")

df_orders = df_orders[df_orders["order_status"] == "delivered"]

df_orders["order_date"] = pd.to_datetime(df_orders["order_date"], format="%Y-%m-%d")

df_sorted = df_orders.sort_values(["customer_id", "order_date"]) # sap xep theo customer

df_sorted["inter_order_gap"] = df_sorted.groupby("customer_id")["order_date"].diff().dt.days # tao cot ngay gap

df_gaps = df_sorted.dropna(subset=["inter_order_gap"]) # xoa null ( tuc la ngay dau tien hoac mua 1 lan)
print(df_gaps[["inter_order_gap"]])

median_gap = df_gaps["inter_order_gap"].median()

print(f"Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) là: {median_gap} ngày")

        inter_order_gap
238890           1101.0
374571            632.0
544446           1037.0
632148           3211.0
24017              89.0
...                 ...
531056            546.0
549990            225.0
557438             48.0
559725             24.0
591841            339.0

[431601 rows x 1 columns]
Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) là: 175.0 ngày


Chon dap an C

Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

In [3]:
import pandas as pd

df_products = pd.read_csv("products.csv")

df_products["gross_margin"] = (df_products["price"] - df_products["cogs"]) / df_products["price"] # tao cot dua tren cong thuc

df_grouped = df_products.groupby("segment")["gross_margin"].mean().reset_index() # group theo segment va tinh mean
df_grouped.columns = ["segment", "avg_gross_margin"]

df_result = df_grouped.sort_values("avg_gross_margin", ascending=False)

top_segment = df_result.iloc[0]["segment"]
top_margin = df_result.iloc[0]["avg_gross_margin"]

print(f"Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất là: {top_segment} với mức {top_margin:.2%}")

Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất là: Standard với mức 31.34%


D

Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [4]:
import pandas as pd

df_returns = pd.read_csv("returns.csv")
df_products = pd.read_csv("products.csv")

df_streetwear = df_products[df_products["category"] == "Streetwear"] # lay streetwear

df_joined = df_returns.merge(df_streetwear, on="product_id", how="inner") # join

df_reasons = df_joined.groupby("return_reason").size().reset_index(name="reason_count") # dem so luong ly do return 

df_top_reason = df_reasons.sort_values("reason_count", ascending=False)

top_reason = df_top_reason.iloc[0]["return_reason"]
top_count = df_top_reason.iloc[0]["reason_count"]

print(f"Lý do trả hàng xuất hiện nhiều nhất cho danh mục Streetwear là: '{top_reason}' với {top_count} bản ghi.")

Lý do trả hàng xuất hiện nhiều nhất cho danh mục Streetwear là: 'wrong_size' với 7626 bản ghi.


B

Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [5]:
import pandas as pd

df_traffic = pd.read_csv("web_traffic.csv")

df_grouped = df_traffic.groupby("traffic_source")["bounce_rate"].mean().reset_index(name="avg_bounce_rate") # gom nhom theo source va tinh mean

df_sorted = df_grouped.sort_values("avg_bounce_rate", ascending=True)

best_source = df_sorted.iloc[0]["traffic_source"]
lowest_bounce_rate = df_sorted.iloc[0]["avg_bounce_rate"]

print(f"Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: '{best_source}' với mức {lowest_bounce_rate:.2%}")

Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: 'email_campaign' với mức 0.45%


C

Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [6]:
import pandas as pd

df_order_items = pd.read_csv("order_items.csv")

total_rows = len(df_order_items)

df_promoted = df_order_items[df_order_items["promo_id"].notnull()] # loc ra cac dong ko null cua promo_id

promoted_rows = len(df_promoted)

percentage = (promoted_rows / total_rows) * 100

print(f"Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi xấp xỉ: {percentage:.2f}%")

C:\Users\minhm\AppData\Local\Temp\ipykernel_24480\1217578240.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_order_items = pd.read_csv("order_items.csv")


Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi xấp xỉ: 38.66%


C

Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)

In [7]:
import pandas as pd

df_customers = pd.read_csv("customers.csv")
df_orders = pd.read_csv("orders.csv")

print(df_customers[df_customers["age_group"].isnull()])

df_cust_filtered = df_customers[df_customers["age_group"].notnull()] #loc null

df_joined = df_cust_filtered.merge(df_orders, on="customer_id", how="left") #join de dem so order theo age

df_grouped = df_joined.groupby("age_group").agg(
    total_orders=("order_id", "count"),
    total_customers=("customer_id", "nunique")
).reset_index()

df_grouped["avg_orders_per_customer"] = df_grouped["total_orders"] / df_grouped["total_customers"]

df_sorted = df_grouped.sort_values("avg_orders_per_customer", ascending=False)

top_age_group = df_sorted.iloc[0]["age_group"]
top_avg = df_sorted.iloc[0]["avg_orders_per_customer"]

print(f"Nhóm tuổi có số đơn hàng trung bình cao nhất là: '{top_age_group}' với {top_avg:.2f} đơn/khách.")

Empty DataFrame
Columns: [customer_id, zip, city, signup_date, gender, age_group, acquisition_channel]
Index: []
Nhóm tuổi có số đơn hàng trung bình cao nhất là: '55+' với 5.41 đơn/khách.


A

Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

In [13]:
import pandas as pd

df_orders = pd.read_csv("orders.csv")
df_order_items = pd.read_csv("order_items.csv")
df_geography = pd.read_csv("geography.csv")

df_order_items["item_revenue"] = df_order_items["unit_price"] * df_order_items["quantity"] # doanh thu = gia sau discount * so luong

df_orders_revenue = df_order_items.groupby("order_id")["item_revenue"].sum().reset_index(name="order_revenue") # tinh tong doanh thu cho rung don hang

df_joined = df_orders.merge(df_orders_revenue, on="order_id", how="inner")
df_final = df_joined.merge(df_geography, on="zip", how="inner")

df_region_revenue = df_final.groupby("region")["order_revenue"].sum().reset_index(name="total_revenue")

df_sorted = df_region_revenue.sort_values("total_revenue", ascending=False)

top_region = df_sorted.iloc[0]["region"]
top_revenue = df_sorted.iloc[0]["total_revenue"]

print(f"Vùng (region) tạo ra tổng doanh thu cao nhất: '{top_region}' với doanh thu: {top_revenue:.2f} .")

C:\Users\minhm\AppData\Local\Temp\ipykernel_24480\181766208.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_order_items = pd.read_csv("order_items.csv")


Vùng (region) tạo ra tổng doanh thu cao nhất: 'East' với doanh thu: 7637532676.20 .


C

Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?

In [9]:
import pandas as pd

df_orders = pd.read_csv("orders.csv")

df_cancelled = df_orders[df_orders["order_status"] == "cancelled"] #loc cancel

df_payment_counts = df_cancelled.groupby("payment_method").size().reset_index(name="cancelled_count") # gom nhom theo phuong thuc

df_sorted = df_payment_counts.sort_values("cancelled_count", ascending=False)

top_payment_method = df_sorted.iloc[0]["payment_method"]
top_count = df_sorted.iloc[0]["cancelled_count"]

print(f"Phương thức thanh toán được sử dụng nhiều nhất trong các đơn hàng bị hủy là: '{top_payment_method}' với {top_count} đơn.")

Phương thức thanh toán được sử dụng nhiều nhất trong các đơn hàng bị hủy là: 'credit_card' với 28452 đơn.


A

Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

In [10]:
import pandas as pd

df_products = pd.read_csv("products.csv")
df_order_items = pd.read_csv("order_items.csv")
df_returns = pd.read_csv("returns.csv")

df_items_joined = df_order_items.merge(df_products, on="product_id", how="inner")
df_orders_count = df_items_joined.groupby("size").size().reset_index(name="total_ordered_rows")

df_returns_joined = df_returns.merge(df_products, on="product_id", how="inner")
df_returns_count = df_returns_joined.groupby("size").size().reset_index(name="total_returned_rows")

df_rates = df_orders_count.merge(df_returns_count, on="size", how="left")

df_rates["return_rate"] = df_rates["total_returned_rows"] / df_rates["total_ordered_rows"]

sizes_to_check = ["S", "M", "L", "XL"]
df_filtered = df_rates[df_rates["size"].isin(sizes_to_check)]
df_sorted = df_filtered.sort_values("return_rate", ascending=False)

top_size = df_sorted.iloc[0]["size"]
top_rate = df_sorted.iloc[0]["return_rate"]

print(f"Kích thước có tỷ lệ trả hàng cao nhất là: '{top_size}' với tỷ lệ {top_rate:.2%}")

C:\Users\minhm\AppData\Local\Temp\ipykernel_24480\3174629649.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_order_items = pd.read_csv("order_items.csv")


Kích thước có tỷ lệ trả hàng cao nhất là: 'S' với tỷ lệ 5.65%


A

Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

In [11]:
import pandas as pd

df_payments = pd.read_csv("payments.csv")

df_grouped = df_payments.groupby("installments")["payment_value"].mean().reset_index(name="avg_payment_value")

df_sorted = df_grouped.sort_values("avg_payment_value", ascending=False)

top_installment = df_sorted.iloc[0]["installments"]
top_value = df_sorted.iloc[0]["avg_payment_value"]

print(f"Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: {top_installment} kỳ với mức {top_value:,.2f}")

Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: 6.0 kỳ với mức 24,446.65


C